# 🖌️ Qwen-Image-Edit 2509 — ComfyUI on Colab (A100) · **Hugging Face weights, Drive LoRAs**

Same workflow as `qwen_image_edit_comfyui_colab.ipynb`, with storage split by file size:

| What | Size | Where it lives | Why |
|---|---|---|---|
| Diffusion model, text encoder, VAE | ~28 GB | **Hugging Face** → runtime local disk, re-pulled each session | Too big to keep in Drive; `hf_transfer` re-fetches all of it in ~3–7 min |
| **LoRAs** (Lightning + your own) | ~850 MB each | **Google Drive**, read in place | Small, often your own — worth keeping durably |
| Edits & input images | small | **Google Drive** | You want to keep these |

Net Drive usage: **your LoRAs and images only.** The 28 GB of base weights never touch it.

Qwen-Image-Edit is an **instruction-driven image editor** — change a background, swap an outfit, restyle a scene, add/remove objects — by describing the edit in plain language. The **2509** build accepts **up to 3 input images** (main image + style / background references). This notebook is wired to the bundled workflow `workflows/qwen_image_edit_2509_subgraph.json`, which uses the **4-step Lightning LoRA** for fast (~4 step) edits.

| Component | File | ComfyUI folder | Source |
|---|---|---|---|
| Diffusion (edit) model | `qwen_image_edit_2509_fp8_e4m3fn.safetensors` (~20 GB) | `diffusion_models` | HF |
| Text encoder | `qwen_2.5_vl_7b_fp8_scaled.safetensors` (~9 GB) | `text_encoders` | HF |
| VAE | `qwen_image_vae.safetensors` (~250 MB) | `vae` | HF |
| Lightning 4-step LoRA | `Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors` (~850 MB) | `loras` | **Drive** |

> The bundled workflow uses only **ComfyUI core nodes** (`UNETLoader`, `CLIPLoader`, `VAELoader`, `LoraLoaderModelOnly`, `TextEncodeQwenImageEditPlus`, `CFGNorm`, `ModelSamplingAuraFlow`, `KSampler`, …). `TextEncodeQwenImageEditPlus` / `CFGNorm` need a **recent ComfyUI** — Step 3 pulls latest, so restart ComfyUI after updates.

**Setup:** `Runtime → Change runtime type → A100 GPU`. The edit model is fp8 (~20 GB) and wants **≥16 GB VRAM** free; A100 (40 GB) is comfortable, L4 (24 GB) works, T4 (16 GB) is borderline — add `--lowvram` in Step 7 if you hit OOM.

## Step 1 — Verify GPU

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu}')
if 'A100' in gpu:
    print('✅ A100 (40 GB) — ideal for Qwen-Image-Edit 2509 fp8')
elif 'L4' in gpu:
    print('✅ L4 (24 GB) — works with the fp8 model; add --lowvram in Step 7 if OOM')
elif 'T4' in gpu:
    print('⚠️  T4 (16 GB) is borderline for the fp8 edit model. Prefer A100/L4, or use --lowvram in Step 7.')
else:
    print('⚠️  Recommended: A100. Runtime → Change runtime type → A100 GPU')

## Step 2 — Mount Drive & set paths

Drive holds `loras/`, `output/` and `input_images/`. The ~28 GB of base weights go to
`/content/models` on the runtime's local disk, which is wiped when the runtime recycles —
that's the point: they are re-pulled from Hugging Face instead of occupying Drive.

In [ ]:
import os, shutil
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ComfyUI_Qwen'

# Big weights: local disk, re-fetched from Hugging Face each session.
HF_MODELS_DIR = '/content/models'
# LoRAs: durable, in Drive, read in place (small enough that FUSE reads are fine).
LORA_DIR   = f'{DRIVE_BASE}/models/loras'
OUTPUT_DIR = f'{DRIVE_BASE}/output'
INPUT_DIR  = f'{DRIVE_BASE}/input_images'

HF_SUBDIRS = ('diffusion_models', 'text_encoders', 'vae')

for d in [f'{HF_MODELS_DIR}/{s}' for s in HF_SUBDIRS] + [LORA_DIR, OUTPUT_DIR, INPUT_DIR]:
    os.makedirs(d, exist_ok=True)

free = shutil.disk_usage('/content').free / 1024**3
print('✅ Paths ready')
print(f'   base weights (HF → local) → {HF_MODELS_DIR}')
print(f'   loras        (Drive)      → {LORA_DIR}')
print(f'   output       (Drive)      → {OUTPUT_DIR}')
print(f'   input        (Drive)      → {INPUT_DIR}')
print(f'\n   local disk free: {free:.0f} GB')
if free < 40:
    print('⚠️  The base weights need ~28 GB plus download headroom.')
    print('   Runtime → Disconnect and delete runtime gives you a clean disk.')

## Step 3 — Install ComfyUI + custom nodes

Native Qwen-Image-Edit support is built into recent ComfyUI core. Pulls latest ComfyUI (so `TextEncodeQwenImageEditPlus` / `CFGNorm` are present) and adds ComfyUI-Manager.

In [ ]:
import os, subprocess
os.chdir('/content')

# ComfyUI — clone fresh if missing/corrupt, else update
if os.path.exists('/content/ComfyUI'):
    ok = subprocess.run(['git', 'rev-parse', '--git-dir'], cwd='/content/ComfyUI',
                        capture_output=True).returncode == 0
    if ok:
        !cd /content/ComfyUI && git pull -q
        print('✅ ComfyUI updated')
    else:
        !rm -rf /content/ComfyUI && git clone -q https://github.com/comfyanonymous/ComfyUI.git
        print('✅ ComfyUI re-cloned (was corrupt)')
else:
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
    print('✅ ComfyUI cloned')

# Core requirements (flag avoids reinstalling every session)
if not os.path.exists('/content/comfyui_reqs_installed'):
    !pip install -q -r /content/ComfyUI/requirements.txt
    open('/content/comfyui_reqs_installed', 'w').close()
    print('✅ Requirements installed')
else:
    print('✅ Requirements already installed')

# Custom nodes (ComfyUI-Manager for easy node/model management)
for name, repo in [('ComfyUI-Manager', 'https://github.com/ltdrdata/ComfyUI-Manager.git')]:
    path = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(path):
        !git clone -q {repo} {path}
        if os.path.exists(f'{path}/requirements.txt'):
            !pip install -q -r {path}/requirements.txt
        print(f'✅ {name} installed')
    else:
        !cd {path} && git pull -q
        print(f'✅ {name} ready')

print('\n✅ All installs complete')

## Step 4 — Link both locations into ComfyUI

ComfyUI only ever sees `models/<folder>`, so the two sources can be symlinked side by side:
the three base-weight folders point at local disk, `loras` points at Drive. `output` and
`input` are linked to Drive too, so uploads and edits persist across sessions.

In [ ]:
import os, shutil
COMFY_MODELS = '/content/ComfyUI/models'

links = {f'{COMFY_MODELS}/{s}': f'{HF_MODELS_DIR}/{s}' for s in HF_SUBDIRS}
links[f'{COMFY_MODELS}/loras'] = LORA_DIR          # ← Drive
links['/content/ComfyUI/output'] = OUTPUT_DIR      # ← Drive
links['/content/ComfyUI/input']  = INPUT_DIR       # ← Drive

for dst, src in links.items():
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst):
        if os.path.realpath(dst) == os.path.realpath(src):
            print(f'  ✅ {os.path.basename(dst)} → {src}')
            continue
        os.unlink(dst)
    elif os.path.isdir(dst):
        # Rescue anything ComfyUI already wrote here before replacing the dir.
        for entry in os.listdir(dst):
            target = os.path.join(src, entry)
            if not os.path.exists(target):
                shutil.move(os.path.join(dst, entry), target)
        shutil.rmtree(dst)
    elif os.path.exists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f'  ✅ {os.path.basename(dst)} → {src}')
print('\n✅ Folders linked')

## Step 5 — Pull the base weights from Hugging Face

~28 GB straight to local disk. `hf_transfer` runs this multi-threaded at roughly
100–300 MB/s, so a cold session costs about **3–7 minutes**. Nothing here is written to
Drive. Re-running is free once the files are on disk.

In [ ]:
import os, shutil

# hf_transfer = multi-threaded downloads; it is what makes re-pulling 28 GB each
# session cheap enough to skip a Drive cache entirely.
try:
    import hf_transfer  # noqa: F401
except ImportError:
    !pip install -q hf_transfer
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

EDIT = 'Comfy-Org/Qwen-Image-Edit_ComfyUI'
BASE = 'Comfy-Org/Qwen-Image_ComfyUI'
STAGE = '/content/hf_stage'

MODELS = [
    ('Qwen-Image-Edit 2509 fp8 (~20GB)', EDIT,
     'split_files/diffusion_models/qwen_image_edit_2509_fp8_e4m3fn.safetensors', 'diffusion_models'),
    ('Text Encoder Qwen2.5-VL 7B fp8 (~9GB)', BASE,
     'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', 'text_encoders'),
    ('VAE qwen_image_vae (~250MB)', BASE,
     'split_files/vae/qwen_image_vae.safetensors', 'vae'),
]

for label, repo, remote, folder in MODELS:
    name = os.path.basename(remote)
    dest = f'{HF_MODELS_DIR}/{folder}/{name}'
    if os.path.exists(dest) and os.path.getsize(dest) > 1024:
        print(f'  ✅ Ready ({os.path.getsize(dest)/1024**3:.2f}GB): {label}')
        continue
    print(f'  ⬇️  Downloading: {label}')
    try:
        tmp = hf_hub_download(repo_id=repo, filename=remote, local_dir=STAGE)
    except Exception as e:
        print(f'     hf_hub_download failed ({e}); falling back to wget')
        tmp = f'{STAGE}/{name}'
        os.makedirs(STAGE, exist_ok=True)
        url = f'https://huggingface.co/{repo}/resolve/main/{remote}'
        !wget -q --show-progress -O "{tmp}" "{url}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.move(tmp, dest)   # instant: STAGE and HF_MODELS_DIR are both on local disk
    print(f'  ✅ Done ({os.path.getsize(dest)/1024**3:.2f}GB): {label}')

shutil.rmtree(STAGE, ignore_errors=True)
print('\n✅ Base weights ready at', HF_MODELS_DIR)

## Step 5b — LoRAs — **kept in Drive**

Unlike the base weights, LoRAs are downloaded **once into Drive** and reused every session.
This cell fetches the **Lightning 4-step LoRA** the bundled workflow depends on, plus
anything you list in `EXTRA_LORAS`, and then shows everything in the folder.

Two ways to add your own, both landing in Drive `ComfyUI_Qwen/models/loras/`:

1. **Drop-in:** copy any `.safetensors` LoRA into that Drive folder. Nothing to run.
2. **Auto-download:** add `(url, filename)` entries to `EXTRA_LORAS` below. Works with any
   Hugging Face `resolve/main/...` link.

Then in the workflow, point a `LoraLoaderModelOnly` at it (chain multiple by stacking
`LoraLoaderModelOnly` nodes; keep strengths modest, e.g. 0.6–1.0).

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

LORA_STAGE = '/content/lora_stage'   # download locally, then copy to Drive
os.makedirs(LORA_STAGE, exist_ok=True)

# The Lightning LoRA the bundled 4-step workflow needs.
LIGHTNING = ('lightx2v/Qwen-Image-Lightning',
             'Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors')

# Add your own here → ('https://.../file.safetensors', 'saved_name.safetensors')
EXTRA_LORAS = [
    # ('https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/split_files/loras/Qwen-Image-Edit-2509-Relight.safetensors',
    #  'Qwen-Image-Edit-2509-Relight.safetensors'),
]

def have(p):
    return os.path.exists(p) and os.path.getsize(p) > 1024

repo, remote = LIGHTNING
name = os.path.basename(remote)
dest = f'{LORA_DIR}/{name}'
if have(dest):
    print(f'  ✅ Already in Drive ({os.path.getsize(dest)/1024**2:.0f}MB): {name}')
else:
    print(f'  ⬇️  Downloading: {name}')
    try:
        tmp = hf_hub_download(repo_id=repo, filename=remote, local_dir=LORA_STAGE)
    except Exception as e:
        print(f'     hf_hub_download failed ({e}); falling back to wget')
        tmp = f'{LORA_STAGE}/{name}'
        url = f'https://huggingface.co/{repo}/resolve/main/{remote}'
        !wget -q --show-progress -O "{tmp}" "{url}"
    shutil.copyfile(tmp, dest)       # copy, not move: Drive is a different filesystem
    print(f'  ✅ Saved to Drive ({os.path.getsize(dest)/1024**2:.0f}MB): {name}')

for url, name in EXTRA_LORAS:
    dest = f'{LORA_DIR}/{name}'
    if have(dest):
        print(f'  ✅ Already in Drive ({os.path.getsize(dest)/1024**2:.0f}MB): {name}')
        continue
    print(f'  ⬇️  Downloading: {name}')
    tmp = f'{LORA_STAGE}/{name}'
    !wget -q --show-progress -O "{tmp}" "{url}"
    shutil.copyfile(tmp, dest)
    print(f'  ✅ Saved to Drive ({os.path.getsize(dest)/1024**2:.0f}MB): {name}')

shutil.rmtree(LORA_STAGE, ignore_errors=True)

loras = sorted(f for f in os.listdir(LORA_DIR) if f.endswith('.safetensors'))
size = sum(os.path.getsize(f'{LORA_DIR}/{f}') for f in loras) / 1024**3
print(f'\n✅ {len(loras)} LoRA(s) in Drive ({size:.2f} GB) at {LORA_DIR}')
for f in loras:
    print('   ·', f)
print('\n↻ Restart ComfyUI (re-run Step 7) for new LoRAs to appear in the dropdown.')

## Step 6 — Install the bundled workflow

Copies `workflows/qwen_image_edit_2509_subgraph.json` (this repo) into ComfyUI's user workflows so it appears under **Workflows** (📂 sidebar) in the UI. Clones this repo into Colab if needed.

In [ ]:
import os, shutil

REPO_URL = 'https://github.com/mmorrisj/qwen_edit.git'
REPO_DIR = '/content/qwen_edit'
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q

src = f'{REPO_DIR}/workflows/qwen_image_edit_2509_subgraph.json'
wf_dir = '/content/ComfyUI/user/default/workflows'
os.makedirs(wf_dir, exist_ok=True)
if os.path.exists(src):
    shutil.copy(src, f'{wf_dir}/qwen_image_edit_2509_subgraph.json')
    print('✅ Workflow installed → open it from the Workflows (📂) sidebar in ComfyUI')
else:
    print('⚠️  Workflow file not found in repo; load it manually via Workflow → Open.')

## Step 7 — Launch ComfyUI + public URL

Starts ComfyUI, waits until it's actually serving, then exposes it. Pick a `TUNNEL` method:

- **`colab`** *(default — no auth)*: opens ComfyUI as a **clickable "new window" link + an embedded iframe** inside this cell's output. ⚠️ Do **not** copy the raw `...prod.colab.dev` URL into a separate browser — it only works inside this Colab session and otherwise returns **HTTP 404**. Use the link/iframe this cell renders.
- **`cloudflare`**: quick `trycloudflare.com` tunnel that works in any browser. The cell first deletes any stale `~/.cloudflared` credentials (the usual cause of *"authentication"* errors) and installs a fresh binary.
- **`ngrok`**: paste a free authtoken from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).

If ComfyUI itself fails to start, the tail of its log is shown. **Keep this cell running.**

In [ ]:
#@title Step 7 — Launch ComfyUI + tunnel { display-mode: "form" }
TUNNEL = "colab"  #@param ["colab", "cloudflare", "ngrok"]
NGROK_TOKEN = ""  #@param {type:"string"}

import os, re, time, subprocess, urllib.request

PORT = 8188
LOG = '/tmp/comfyui.log'

# --- start ComfyUI (kill any previous run first) ---
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# --enable-cors-header '*' relaxes cross-origin checks for proxied access.
# Add '--lowvram' to the list below if you hit OOM on a 16-24 GB GPU.
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
     '--preview-method', 'auto', '--enable-cors-header', '*'],
    cwd='/content/ComfyUI', stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

print('Waiting for ComfyUI to start...')
ready = False
for _ in range(60):
    time.sleep(2)
    if comfy.poll() is not None:
        print('\n❌ ComfyUI exited. Last log lines:\n')
        print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
        raise SystemExit('ComfyUI failed to start — see log above.')
    try:
        if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2).status == 200:
            ready = True
            break
    except Exception:
        pass
if not ready:
    print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
    raise SystemExit('ComfyUI did not become ready in time — see log above.')
print('✅ ComfyUI is serving on :%d' % PORT)

def banner(url, note=''):
    print('\n' + '=' * 64)
    print('  🚀  Open ComfyUI:  ' + url)
    if note:
        print('  ' + note)
    print('=' * 64)

tunnel = None

if TUNNEL == 'colab':
    # Same-origin embed/link — avoids both the 404 (raw proxy URL pasted in a new
    # browser) and ComfyUI's 403 host/origin check. RECOMMENDED on Colab.
    from google.colab import output
    print('▶ Click this link to open ComfyUI in a new tab:')
    output.serve_kernel_port_as_window(PORT)
    print('\n▶ ...or use ComfyUI embedded right here:')
    output.serve_kernel_port_as_iframe(PORT, height='820')

elif TUNNEL == 'ngrok':
    # ngrok forwards Host == its own domain == Origin, so ComfyUI's host/origin
    # check passes. Reliable public URL that works in any browser.
    if not NGROK_TOKEN:
        raise SystemExit('Set NGROK_TOKEN in the form (free at dashboard.ngrok.com), or use TUNNEL="colab".')
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    banner(ngrok.connect(PORT, 'http').public_url, '(ngrok)')

elif TUNNEL == 'cloudflare':
    # Quick tunnel. ComfyUI v1.19+ may 403 ("non matching host and origin")
    # through a proxy; --http-host-header keeps the forwarded Host aligned, and a
    # stale cookie is the other common cause — open the link in an incognito tab.
    # If it still 403s, switch TUNNEL to "ngrok" or "colab".
    !rm -rf ~/.cloudflared
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate',
         '--url', f'http://127.0.0.1:{PORT}', '--http-host-header', f'127.0.0.1:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_re = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
    found = False
    t0 = time.time()
    for line in tunnel.stdout:
        m = url_re.search(line)
        if m:
            banner(m.group(0), '403? open in an incognito tab; still 403 → use TUNNEL="ngrok" or "colab"')
            found = True
            break
        if time.time() - t0 > 40:
            break
    if not found:
        print('⚠️  Cloudflare returned no URL. Set TUNNEL="colab" or "ngrok" and re-run.')

print('\n⏳ Keep this cell running. Interrupt to stop.')
try:
    comfy.wait()
except KeyboardInterrupt:
    comfy.terminate()
    if tunnel:
        tunnel.terminate()
    print('\n🛑 ComfyUI stopped')

## Step 8 — Run the edit workflow

In the ComfyUI tab:
1. **Open the workflow:** Workflows (📂 sidebar) → `qwen_image_edit_2509_subgraph`. *(Or `Workflow → Browse Templates → Image → Qwen-Image-Edit`.)*
2. Confirm the loaders inside the **Qwen Image Edit 2509** subgraph point at your files: `UNETLoader` = `qwen_image_edit_2509_fp8_e4m3fn.safetensors`, `CLIPLoader` = `qwen_2.5_vl_7b_fp8_scaled` (type `qwen_image`), `VAELoader` = `qwen_image_vae.safetensors`, `LoraLoaderModelOnly` = `Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors`.
3. **Load Image** node(s) = your source image(s). Upload via the UI, or drop files in Drive `ComfyUI_Qwen/input_images/` (see utility below). 2509 accepts **up to 3** — main image + optional references.
4. Write the edit in the prompt, e.g. *"Remove the yellow balloon"*, *"Change the balloon's color to blue"*, *"Replace the man with a child, keep the same oil-painting style"*.
5. **Settings (with the 4-step Lightning LoRA, as bundled):** steps **4**, CFG **1.0**, sampler **euler**, scheduler **simple**. *For max quality without the LoRA:* bypass the `LoraLoaderModelOnly`, then steps **20**, CFG **2.5** (fp8) or steps **50**, CFG **4.0** (bf16).
6. **Run.** The edited image lands in Drive `ComfyUI_Qwen/output/`.

**OOM?** Add `--lowvram` to the launch command in Step 7, or shrink the input via the `ImageScaleToTotalPixels` node already in the graph.

---
## 🔧 Utilities

### Copy an input image into ComfyUI

In [ ]:
import os
SOURCE = f'{INPUT_DIR}/my_image.png'  # update filename
if os.path.exists(SOURCE):
    # /content/ComfyUI/input is symlinked to Drive input_images, so it is already
    # visible in the Load Image node. This just confirms the file is present.
    print(f'✅ Available in Load Image node: {os.path.basename(SOURCE)}')
else:
    print(f'⚠️  Not found: {SOURCE}\n   Upload to {INPUT_DIR} (or use the UI upload button)')

### List recent edits (outputs)

In [ ]:
import glob, os
outs = sorted(glob.glob(f'{OUTPUT_DIR}/*.png'), key=os.path.getmtime, reverse=True)
print(f'Found {len(outs)} output image(s):')
for p in outs[:10]:
    print(f'  {os.path.getsize(p)/1024:.0f} KB  {os.path.basename(p)}')
if outs:
    try:
        from IPython.display import Image, display
        print('\nMost recent:')
        display(Image(outs[0]))
    except Exception:
        pass

### Reclaim Drive space (delete base weights from Drive — LoRAs are kept)

If you ran the original notebook, ~28 GB of base weights are still in
`MyDrive/ComfyUI_Qwen/models`. This notebook no longer needs them. Lists them with sizes;
set `CONFIRM_DELETE = True` to remove them. **`loras/` is never touched**, and neither are
`output/` or `input_images/`.

In [ ]:
#@title Reclaim Drive space { display-mode: "form" }
CONFIRM_DELETE = False  #@param {type:"boolean"}

import os
drive_models = f'{DRIVE_BASE}/models'
keep = os.path.realpath(LORA_DIR)
exts = ('.safetensors', '.ckpt', '.pt', '.bin')

found = []
for r, _, fs in os.walk(drive_models):
    if os.path.realpath(r) == keep or os.path.realpath(r).startswith(keep + os.sep):
        continue                                  # never touch LoRAs
    found += [(os.path.join(r, f), os.path.getsize(os.path.join(r, f)))
              for f in fs if f.endswith(exts)]

total = sum(s for _, s in found)
for p, s in sorted(found, key=lambda x: -x[1]):
    print(f'  {s/1024**3:7.2f} GB  {os.path.relpath(p, drive_models)}')
print(f'\n  {total/1024**3:7.2f} GB  TOTAL deletable (LoRAs excluded)')

if not found:
    print('\n✅ No base weights left in Drive.')
elif CONFIRM_DELETE:
    for p, _ in found:
        os.remove(p)
    print(f'\n✅ Deleted {len(found)} files — {total/1024**3:.1f} GB freed.')
    print('   Empty Drive Trash to actually reclaim the quota.')
else:
    print('\n(dry run — tick CONFIRM_DELETE to delete these files)')

---
## 📋 Prompt & troubleshooting tips

**Editing prompts** — describe the change, not the whole scene. Qwen-Image-Edit follows edit instructions well:
- Object: `"remove the yellow balloon"`, `"add a small dog next to the man"`
- Recolor / material: `"change the balloon's color to reflective blue"`
- Restyle: `"convert to a realistic photo, keep composition"`, `"make it an oil painting"`
- Background: `"replace the background with a sunset beach"`
- Text: Qwen is strong at rendering/editing text — `"change the sign to read 'OPEN'"`
- Multi-image (2509): image 1 = subject, image 2 = style/background reference, image 3 = extra reference

**Troubleshooting**
- **`TextEncodeQwenImageEditPlus` / `CFGNorm` missing** → ComfyUI is out of date. Re-run Step 3 (`git pull`), then restart ComfyUI (re-run Step 7).
- **Cloudflare "authentication" / 403** → use `TUNNEL = "colab"` in Step 7 (no auth), or open the Cloudflare link in an incognito tab. The Cloudflare path also wipes stale `~/.cloudflared` creds.
- **OOM (out of memory)** → add `--lowvram` to the launch command in Step 7; keep input images ≤ ~1–2 MP (the `ImageScaleToTotalPixels` node caps this).
- **Wrong CLIP type** → `CLIPLoader` type must be `qwen_image`.
- **Cell stops immediately** → Step 7 prints the ComfyUI log tail on failure; read it for the real error.

**Storage**
- **`No space left on device` in Step 5** → the base weights need ~28 GB on local disk plus download headroom. `Runtime → Disconnect and delete runtime` gives a clean disk.
- **Step 5 re-downloads every session** → expected. `/content` is wiped when the runtime recycles; that's the trade for keeping 28 GB out of Drive. Within one session it's a no-op.
- **A LoRA doesn't appear in ComfyUI** → it must be in `MyDrive/ComfyUI_Qwen/models/loras/`; re-run Step 4, then restart ComfyUI (re-run Step 7).
- **`A Google Drive quota has been exceeded`** → only LoRAs and images come from Drive here, so this is rare; if it happens, wait a few minutes rather than re-running Step 5.
- **Want the base weights durable?** → keep using the original `qwen_image_edit_comfyui_colab.ipynb`, which caches everything in Drive.